## 사용 사례: Hybrid Deployment 아키텍처

### 시나리오: 인터넷 액세스를 통한 안전한 데이터 처리

이 deployment pattern에서는 보안과 기능의 균형을 위해 AgentCore component를 분리합니다.

**AgentCore Browser (Public Mode)**
- 인터넷 액세스가 가능한 public subnet에 배포
- 공개 웹 사이트 및 서비스 탐색 지원
- 외부 통합 및 실시간 데이터 검색 관리
- 직접 인터넷 연결을 활용한 실시간 작업 수행

**AgentCore Runtime (VPC Mode)**
- 안전한 VPC 내부의 private subnet에 배포
- 민감한 데이터 및 비즈니스 로직 처리
- 엄격한 네트워크 격리 유지
- 안전한 내부 채널을 통해 브라우저 component와 통신

### 이점

- **보안**: 민감한 처리 작업을 private network에 격리
- **성능**: NAT overhead 없이 브라우저 작업에서 인터넷에 직접 액세스
- **규정 준수**: 데이터 격리에 관한 규제 요구 사항 충족
- **확장성**: Workload 요구에 따라 각 component를 독립적으로 확장

### 아키텍처 흐름

![image.png](architecture-browser.png)

# CloudFormation Stack 실행 지침

## 사전 요구 사항
- 적절한 권한으로 구성된 AWS CLI
- 준비된 CloudFormation template file

## 실행 단계

### 1. 아래 단계를 실행해 CFN 시작(약 10분)

In [ ]:
import boto3

# 구성 변수
region = "us-east-1"  # 원하는 리전으로 변경
template_path = "cfn-browser.yaml"
stack_name = "browser-stack"

# 구성 가능한 리전으로 CloudFormation client 초기화
cf_client = boto3.client("cloudformation", region_name=region)

# CloudFormation template 읽기
with open(template_path, "r") as template_file:
    template_body = template_file.read()

try:
    # CloudFormation stack 생성
    response = cf_client.create_stack(
        StackName=stack_name,
        TemplateBody=template_body,
        Capabilities=["CAPABILITY_IAM", "CAPABILITY_NAMED_IAM"],
    )

    print(f"Stack creation initiated in region: {region}")
    print(f"Stack ID: {response['StackId']}")

    # Stack 생성이 완료될 때까지 대기
    waiter = cf_client.get_waiter("stack_create_complete")
    print("Waiting for stack creation to complete...")
    waiter.wait(StackName=stack_name)

    print(f"Stack '{stack_name}' created successfully in {region}!")

except Exception as e:
    print(f"Error creating stack: {str(e)}")

### 2. 테스트 지침

In [ ]:
import boto3
from IPython.display import Markdown


# CloudFormation output에서 AgentRuntime ARN 가져오기
def get_cfn_output(stack_name, output_key, region="us-east-1"):
    """CloudFormation 스택의 출력 값을 가져옵니다."""
    cfn = boto3.client("cloudformation", region_name=region)
    try:
        response = cfn.describe_stacks(StackName=stack_name)
        outputs = response["Stacks"][0]["Outputs"]
        for output in outputs:
            if output["OutputKey"] == output_key:
                return output["OutputValue"]
    except Exception as e:
        print(f"Error fetching CFN output: {e}")
    return None


# 이전 셀의 stack_name을 사용해 AgentRuntime ARN 가져오기
agent_runtime_arn = get_cfn_output(stack_name, "AgentRuntimeArn")
agent_runtime_id = get_cfn_output(stack_name, "AgentRuntimeId")
development_instance = get_cfn_output(stack_name, "DevelopmentInstanceId")


# 전체 테스트 지침 생성
instructions = f"""# Bedrock Agent Runtime 테스트 지침

## 사전 요구 사항
- CloudFormation stack 생성이 성공적으로 완료됨

## 단계별 테스트 과정

### 1. EC2 instance에 연결
Browser Connector 또는 SSH를 통해 EC2 instance `{development_instance}`에 연결합니다.

### 2. 환경 설정
EC2 instance에서 다음 명령을 실행합니다.

```bash
sudo yum update -y

# 개발 도구 설치
sudo dnf install git -y && \\
curl -LsSf https://astral.sh/uv/install.sh | sh && \\
echo 'export PATH="$HOME/.cargo/bin:$PATH"' >> ~/.bashrc && \\
source ~/.bashrc && \\
echo 'Install Python Venv'

uv init vpc-browser --python 3.13 && cd vpc-browser
uv venv --python 3.13
source .venv/bin/activate
uv pip install boto3

cat > call-agent.py << 'EOF'
import boto3
import json

client = boto3.client('bedrock-agentcore', region_name='us-east-1')

payload = json.dumps({{
    "prompt": "What is the Weather in Richmond VA Today?"
}})

response = client.invoke_agent_runtime(
    agentRuntimeArn="{agent_runtime_arn}",
    runtimeSessionId='dfmeoagmreaklgmrkleafremoigrmtesogmtrskhmtkrlshmt',  # 33자 이상이어야 함
    payload=payload,
    qualifier="DEFAULT"  # 선택 사항
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)
EOF


```

### 3. Agent 테스트 실행
```
python call-agent.py
```

### 4. CloudWatch에서 로그 모니터링
- AWS Console에서 CloudWatch Logs로 이동합니다.
- Log group `/aws/bedrock-agentcore/runtimes/{agent_runtime_id}`를 찾습니다.
- Live browsing을 보려면 Console -> Amazon Bedrock AgentCore -> Built-in Tools -> Browser tools -> browser_stack_browser -> View live session으로 이동합니다.
- 실시간 실행 로그와 오류를 모니터링합니다.

"""

Markdown(instructions)

### 3. 정리

In [ ]:
import boto3

# Stack 삭제
cfn = boto3.client("cloudformation", region_name=region)
cfn.delete_stack(StackName=stack_name)

print(f"Stack '{stack_name}' deletion initiated in region '{region}'")

# 삭제가 완료될 때까지 대기
waiter = cfn.get_waiter("stack_delete_complete")
print("Waiting for stack deletion to complete...")
waiter.wait(StackName=stack_name)